In [1]:
from pathlib import Path
resultsdirs = {
    # "OracleSAD": Path("/data4/Henri/j3/framewiseSpeakerCounting/results/J2_RUN/J2_BXLS_main_exp/oracle"),
    "OracleSAD": Path("/data4/Henri/j3/framewiseSpeakerCounting/results/J2_RUN/Klaus_PALD_3D_1/oracle"),
    # # "COSAD": Path("/data4/Henri/j3/framewiseSpeakerCounting/results/J2_RUN/J2_BXLS_main_exp/PrecomputedSAD"),
    "COSAD": Path("/data4/Henri/j3/framewiseSpeakerCounting/results/J2_RUN/Klaus_PALD_3D_1/PrecomputedSAD"),
}

In [2]:
# The first step is to load all csv files in resultsdir and concatenate them into a single dataframe
# important: every method has its own csv file and the method name is part of the filename but not the content
# I provide a list of the method names to load
method_names_long = [
  # "online-cgmm-mvdr",
  "BlockOnlineGSS_bl4_pc300",
  "RTF_Estimator_C",
  "RTF_Estimator_CSn",
  "RTF_Estimator_CWn",
  "RTF_Estimator_CSv",
  "RTF_Estimator_CWv",
  "RTF_Estimator_BOP", 
  "RTF_Estimator_BOP_S", 
  "RTF_Estimator_BOP_W", 
  "RTF_Estimator_BOPO", 
  "RTF_Estimator_BOPO_S", 
  "RTF_Estimator_BOPO_W",
  "RTF_Estimator_CB",
  "RTF_Estimator_CB_S",
  "RTF_Estimator_CB_W",
  "RTF_Estimator_CBW",
  "RTF_Estimator_Oracle", 
]
method_names_short = [
  # "online-cgmm-mvdr",
  "GSS",
  "C",
  "CSn",
  "CWn",
  "CSu",
  "CWu",
  "BOP", 
  "BOP-S", 
  "BOP-W", 
  "BOPO", 
  "BOPO-S", 
  "BOPO-W",
  "CB",
  "CB-S",
  "CB-W",
  "CBW",
  "Oracle", 
]


In [24]:
# Now load the csv files and concatenate them into a single dataframe
import pandas as pd
dfs = {}
for key, resultsdir in resultsdirs.items():
    df_list = []
    for method_short, method_long in zip(method_names_short, method_names_long):
        csv_file = resultsdir / f"{method_long}_results.csv"
        if csv_file.exists():
            # note that the first colum is the scenario_id which should be the index of the df
            df = pd.read_csv(csv_file, index_col=0)
            df["method"] = method_short  # add a column for the method name
            # Append method name to index to ensure uniqueness across the combined dataframe
            df.index = df.index.astype(str) + "_" + method_short
            if method_short == "GSS": # for all columns which have 'SMVDR' in their name make copies of these columns for which the SMVDR is replaced with each of ['MVDR', 'LCMV', 'MPDR', 'LCMP', 'MPDR2', 'LCMP2']
                smvdr_columns = [col for col in df.columns if "SMVDR" in col]
                for col in smvdr_columns:
                    for bf in ['MVDR', 'LCMV', 'MPDR', 'LCMP', 'MPDR2', 'LCMP2']:
                        new_col = col.replace("SMVDR", bf)
                        df[new_col] = df[col]
                
            df_list.append(df)
        else:
            print(f"Warning: {csv_file} does not exist and will be skipped.")
    # Concatenate all dataframes
    df_all = pd.concat(df_list)
    print("Combined dataframe shape:", df_all.shape)
    # print an overview of all columns
    print("Columns in the combined dataframe:", df_all.columns)
    # For all columns with MHA in their name, they are Hermitian Angle metrics up to now in radians,
    # convert them to degrees for better interpretability
    mha_columns = [col for col in df_all.columns if "MHA" in col]
    for col in mha_columns:
        df_all[col] = df_all[col] * (180.0 / 3.141592653589793)  # radians to degrees
    df_all["input_snr_interval"] = df_all["input_snr"].apply(lambda x: f"{(x//5)*5}-{((x//5)+1)*5}")  # create a new column for SNR intervals of 5 dB
    df_all["input_snr_5dBsteps"] = df_all["input_snr"].apply(lambda x: ((x+2.5)//5)*5)  # create a new column for SNR intervals of 5 dB
    dfs[key] = df_all

Combined dataframe shape: (1800, 357)
Columns in the combined dataframe: Index(['MHA', 'WMHA', 'MHA_A1', 'WMHA_A1', 'MHA_A2', 'WMHA_A2', 'MHA_A3',
       'WMHA_A3', 'MHA_D1', 'WMHA_D1',
       ...
       'SISDRo_LCMV_D1', 'DSISDR_LCMV_D1', 'SISDRo_MVDR_D1', 'DSISDR_MVDR_D1',
       'SISDRi_D2', 'SISDRo_LCMV_D2', 'DSISDR_LCMV_D2', 'SISDRo_MVDR_D2',
       'DSISDR_MVDR_D2', 'method'],
      dtype='object', length=357)
Combined dataframe shape: (1200, 357)
Columns in the combined dataframe: Index(['MHA', 'WMHA', 'MHA_A1', 'WMHA_A1', 'MHA_A2', 'WMHA_A2', 'MHA_A3',
       'WMHA_A3', 'MHA_D1', 'WMHA_D1',
       ...
       'SISDRo_LCMV_D1', 'DSISDR_LCMV_D1', 'SISDRo_MVDR_D1', 'DSISDR_MVDR_D1',
       'SISDRi_D2', 'SISDRo_LCMV_D2', 'DSISDR_LCMV_D2', 'SISDRo_MVDR_D2',
       'DSISDR_MVDR_D2', 'method'],
      dtype='object', length=357)


In [23]:
x = 7.4
fun = lambda x: f"{(x//5)*5}-{((x//5)+1)*5}"
fun2 = lambda x: ((x+2.5)//5)*5
print(fun(x))
print(fun2(x))

5.0-10.0
5.0


In [14]:
dfs["OracleSAD"]["input_snr"]

scenario_id
Klaus_PALD_3D_1_test_generator_0_BOPO-W      13.122635
Klaus_PALD_3D_1_test_generator_1_BOPO-W      11.664203
Klaus_PALD_3D_1_test_generator_10_BOPO-W     13.996139
Klaus_PALD_3D_1_test_generator_100_BOPO-W    14.528724
Klaus_PALD_3D_1_test_generator_101_BOPO-W    12.256823
                                               ...    
Klaus_PALD_3D_1_test_generator_95_Oracle     14.906479
Klaus_PALD_3D_1_test_generator_96_Oracle     16.020647
Klaus_PALD_3D_1_test_generator_97_Oracle     18.221700
Klaus_PALD_3D_1_test_generator_98_Oracle     15.809609
Klaus_PALD_3D_1_test_generator_99_Oracle     17.378213
Name: input_snr, Length: 1800, dtype: float64

In [15]:
dfs["OracleSAD"]["input_snr_interval"]

scenario_id
Klaus_PALD_3D_1_test_generator_0_BOPO-W      10.0-15.0
Klaus_PALD_3D_1_test_generator_1_BOPO-W      10.0-15.0
Klaus_PALD_3D_1_test_generator_10_BOPO-W     10.0-15.0
Klaus_PALD_3D_1_test_generator_100_BOPO-W    10.0-15.0
Klaus_PALD_3D_1_test_generator_101_BOPO-W    10.0-15.0
                                               ...    
Klaus_PALD_3D_1_test_generator_95_Oracle     10.0-15.0
Klaus_PALD_3D_1_test_generator_96_Oracle     15.0-20.0
Klaus_PALD_3D_1_test_generator_97_Oracle     15.0-20.0
Klaus_PALD_3D_1_test_generator_98_Oracle     15.0-20.0
Klaus_PALD_3D_1_test_generator_99_Oracle     15.0-20.0
Name: input_snr_interval, Length: 1800, dtype: object

In [9]:
dfs["OracleSAD"]
# print a list of unique metric names. All columns are somehow metrics but many of them are just the same metric with onther beaformer or for another segment. So for each metric i only wnat to consider the first part until the first underscore and i want to have a unique list of these metric names
metric_names = set()
for col in dfs["OracleSAD"].columns:
    metric_name = col.split("_")[0]  # get the part before the first underscore
    metric_names.add(metric_name)
print("Unique metric names:", sorted(metric_names))

Unique metric names: ['DFWSSNR', 'DHGSDR', 'DPESQ', 'DSDR', 'DSINR', 'DSIR', 'DSISDR', 'DSNR', 'DSTOI', 'Di', 'Dn', 'Dt', 'FWSSNRi', 'FWSSNRo', 'Gi', 'Gn', 'Gt', 'HGSDRi', 'HGSDRo', 'MHA', 'PESQi', 'PESQo', 'SDRi', 'SDRo', 'SINRi', 'SINRo', 'SIRi', 'SIRo', 'SISDRi', 'SISDRo', 'SNRi', 'SNRo', 'STOIi', 'STOIo', 'WMHA', 'fixed', 'input', 'method']


In [15]:
# print all columns in the dataframe, really all not printing dots
pd.set_option('display.max_columns', None)
print("Columns in the combined dataframe:", df_all.columns.tolist()[10:30])
# print all columns which have a 'SMVDR' in their name
smvdr_columns = [col for col in dfs["OracleSAD"].columns if "SMVDR" in col]
print("Columns with 'SMVDR' in their name:", smvdr_columns)
# print the dataframe with only the columns which have 'SMVDR' in their name
print(dfs["OracleSAD"][smvdr_columns])

Columns in the combined dataframe: ['MHA_D2', 'WMHA_D2', 'SINRi', 'SINRo_LCMV', 'DSINR_LCMV', 'SINRo_MVDR', 'DSINR_MVDR', 'SIRi', 'SIRo_LCMV', 'DSIR_LCMV', 'SIRo_MVDR', 'DSIR_MVDR', 'SNRi', 'SNRo_LCMV', 'DSNR_LCMV', 'SNRo_MVDR', 'DSNR_MVDR', 'HGSDRi', 'HGSDRo_LCMV', 'DHGSDR_LCMV']
Columns with 'SMVDR' in their name: []
Empty DataFrame
Columns: []
Index: [Klaus_PALD_3D_1_test_generator_0_BOPO-W, Klaus_PALD_3D_1_test_generator_1_BOPO-W, Klaus_PALD_3D_1_test_generator_10_BOPO-W, Klaus_PALD_3D_1_test_generator_100_BOPO-W, Klaus_PALD_3D_1_test_generator_101_BOPO-W, Klaus_PALD_3D_1_test_generator_102_BOPO-W, Klaus_PALD_3D_1_test_generator_103_BOPO-W, Klaus_PALD_3D_1_test_generator_104_BOPO-W, Klaus_PALD_3D_1_test_generator_105_BOPO-W, Klaus_PALD_3D_1_test_generator_106_BOPO-W, Klaus_PALD_3D_1_test_generator_107_BOPO-W, Klaus_PALD_3D_1_test_generator_108_BOPO-W, Klaus_PALD_3D_1_test_generator_109_BOPO-W, Klaus_PALD_3D_1_test_generator_11_BOPO-W, Klaus_PALD_3D_1_test_generator_110_BOPO-W, Klau

In [5]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd

# 1. defined dimensions from your LaTeX document
latex_column_width_pt = 252.0
latex_font_size_pt = 8.0

# 2. Convert to inches for matplotlib
inches_per_pt = 1.0 / 72
fig_width_in = latex_column_width_pt * inches_per_pt
golden_ratio = (5**0.5 - 1) / 2  # approx 0.618
fig_height_in = fig_width_in * golden_ratio #(9 / 16) # 16:9 ratio

# 3. Update rcParams to match LaTeX font settings precisely
mpl.use("pgf")
mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",
    'font.family': 'serif',
    'text.usetex': True,
    'pgf.rcfonts': False,
    'pgf.preamble': r'\usepackage{amsmath,graphicx,siunitx,bm}',
    
    # Font sizes
    'font.size': latex_font_size_pt,
    'axes.labelsize': latex_font_size_pt,      # Axis labels same as text
    'axes.titlesize': latex_font_size_pt,      # Title same as text
    'legend.fontsize': latex_font_size_pt, # Legend smaller (8pt) to fit
    'xtick.labelsize': latex_font_size_pt, # Ticks smaller (8pt)
    'ytick.labelsize': latex_font_size_pt,
    
    # Figure size
    'figure.figsize': [fig_width_in, fig_height_in],
})

def plot_metric_by_group(
    df,
    metric,
    group_by,
    xlabel,
    title,
    ylabel,
    output_file,
):
    # Create figure with the exact calculated size
    fig, ax = plt.subplots(figsize=(fig_width_in, fig_height_in), dpi=600)
    
    sns.barplot(
        data=df,
        x=group_by,
        y=metric,
        hue="method",
        errorbar="se",
        estimator=pd.Series.median,
        capsize=0.05,
        palette="colorblind", # Correct parameter
        ax=ax,
        linewidth=0.15,
    )
    
    ax.grid(True, axis='y', linestyle='--', alpha=0.7, linewidth=0.5)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel(xlabel)
    
    # Legend handling for small figures:
    # Placing it outside on the right makes the figure too wide for a single column.
    # Placing it on top is usually safer for column-width figures.
    ax.legend(
        title=r"\textbf{Method}", 
        loc='upper center', 
        bbox_to_anchor=(0.5, 1.35), # Move up above title
        ncol=4,                     # Spread horizontally
        frameon=False,
        columnspacing=1.0,
        handletextpad=0.2
    )
    
    # Use standard margins instead of tight_layout if you want exact dimensions,
    # but tight_layout is often safer for labels not getting cut off.
    # We use pad=0.2 to minimize wasted white space
    plt.tight_layout(pad=0.2)
    
    plt.savefig(output_file)
    plt.close() # Close to free memory
    
# Now create plots for different metrics
metrics_to_plot = [
    ("WMHA", r"WMHA [°]", r"Utterance Level"),
    ("DSINR", r"SINR Improvement [dB]", r"Utterance Level"),
    ("STOIo", r"STOI", r"Utterance Level"),
    ("WMHA_A2", r"WMHA [°]", r"Segment A2"),
    ("WMHA_A3", r"WMHA [°]", r"Segment A3"),
    ("DSINR_A2", r"SINR Improvement [dB]", r"Segment A2"),
    ("DSINR_A3", r"SINR Improvement [dB]", r"Segment A3"),
    ("STOIo_A2", r"STOI", r"Segment A2"),
    ("STOIo_A3", r"STOI", r"Segment A3"),
]

for metric, ylabel, title in metrics_to_plot:
    # Ensure resultsdir is defined previously
    output_path = resultsdirs["OracleSAD"] / "plots" / f"{metric}_by_input_snr.png"
    # Ensure directory exists
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    plot_metric_by_group(
        dfs["OracleSAD"].reset_index(drop=True),
        metric=metric,
        group_by="input_snr",
        xlabel="Input SNR [dB]",
        title=title,
        ylabel=ylabel,
        output_file=output_path,
    )

In [12]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
from pathlib import Path

# 1. defined dimensions from your LaTeX document
latex_two_column_width_pt = 516.0
latex_font_size_pt = 8.0

# 2. Convert to inches for matplotlib
inches_per_pt = 1.0 / 72
fig_width_in = latex_two_column_width_pt * inches_per_pt 
golden_ratio = (5**0.5 - 1) / 2
fig_height_in = fig_width_in * golden_ratio

# 3. Update rcParams
mpl.use("pgf")
mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",
    'font.family': 'serif',
    'text.usetex': True,
    'pgf.rcfonts': False,
    'pgf.preamble': r'\usepackage{amsmath,graphicx,siunitx,bm}',
    'font.size': latex_font_size_pt,
    'axes.labelsize': latex_font_size_pt,      
    'axes.titlesize': latex_font_size_pt,      
    'legend.fontsize': latex_font_size_pt, 
    'xtick.labelsize': latex_font_size_pt, 
    'ytick.labelsize': latex_font_size_pt,
    'figure.figsize': [fig_width_in, fig_height_in],
})

def plot_2x2_grid_twocol(
    df,
    metrics_list, 
    group_by,
    xlabel_text,
    output_files,
):
    if len(metrics_list) > 4:
        print(f"Warning: metrics_list has {len(metrics_list)} entries. Truncating to first 4.")
        metrics_list = metrics_list[:4]
    
    # --- 1. Prepare Consistent Colors & Order ---
    all_methods = [m for m in ["GSS", "CWu", "BOP", "BOP-S", "BOP-W", "BOPO", "BOPO-S", "BOPO-W", "Oracle"]]
    all_methods = method_names_short # Added for J2
    
    palette_colors = sns.color_palette("colorblind", n_colors=len(all_methods))
    method_color_map = dict(zip(all_methods, palette_colors))
    
    # Identify if we are in the COSAD Scenario where GSS should be hidden
    # Use the first output file to check for filename pattern
    if isinstance(output_files, list) and len(output_files) > 0:
        check_path = output_files[0]
    else:
        check_path = output_files
    is_cosad_plot = "COSAD" in str(check_path)

    # Create figure
    fig, axes = plt.subplots(2, 2, figsize=(fig_width_in, fig_height_in), dpi=600)
    axes_flat = axes.flatten()
    letters = ["(a)", "(b)", "(c)", "(d)"]
    min_medians = []

    for i, (metric, ylabel, title) in enumerate(metrics_list):
        ax = axes_flat[i]
        is_bottom = i >= 2
    
        # --- 2. Conditional Filtering ---
        # Exclude GSS if COSAD
        if is_cosad_plot:
            current_df = df[df["method"] != "GSS"]
        else:
            current_df = df

        # Additional Filtering for MHA (Oracle SAD or COSAD doesn't matter, MHA excludes GSS/Oracle)
        if "MHA" in metric:
            plot_df = current_df[~current_df["method"].isin(["GSS", "Oracle"])]
        else:
            plot_df = current_df

        if not plot_df.empty:
            medians = plot_df.groupby([group_by, "method"])[metric].median()
            if not medians.empty:
                min_medians.append(medians.min())
            else:
                min_medians.append(0)
        else:
            min_medians.append(0)

        current_methods = [m for m in all_methods if m in plot_df["method"].unique()]

        sns.barplot(
            data=plot_df,
            x=group_by,
            y=metric,
            hue="method",
            hue_order=current_methods,
            palette=method_color_map,
            errorbar="se",
            estimator=pd.Series.median,
            ax=ax,
            linewidth=0.15,
        )
        
        ax.grid(True, axis='y', linestyle='--', alpha=0.7, linewidth=0.5)
        ax.set_title(title, pad=3) 
        ax.set_ylabel(ylabel)
        
        if is_bottom:
            ax.set_xlabel(xlabel_text)
        else:
            ax.set_xlabel("")
            ax.set_xticklabels([]) 
            
        ax.text(-0.06, 1.1, r"{" + letters[i] + "}", 
                transform=ax.transAxes, va='top', ha='right', fontsize=latex_font_size_pt)

        if ax.get_legend():
            ax.get_legend().remove()

        # Legend Logic (Index 0)
        if i == 0:
            legend_handles = []
            
            # Iterate through ALL methods to maintain grid positions
            for m in all_methods:
                if is_cosad_plot and m in ["GSS", "Oracle"]:
                    # Create an invisible patch for GSS (preserves the grid slot)
                    # Label is empty or spaces to keep alignment without showing text
                    legend_handles.append(mpatches.Patch(color="none", label="", alpha=0))
                else:
                    legend_handles.append(mpatches.Patch(color=method_color_map[m], label=m))
            
            ax.legend(
                handles=legend_handles,
                loc='upper right', 
                ncol=3, 
                frameon=True,
                columnspacing=0.5,
                handletextpad=0.2,
                labelspacing=0.2,
                fontsize=latex_font_size_pt
            )

    # --- Shared Y-Axis Scaling ---
    r1_min_data = min(min_medians[0], min_medians[1])
    r1_ymin = np.floor(r1_min_data - 1)
    r1_ymax = max(axes_flat[0].get_ylim()[1], axes_flat[1].get_ylim()[1])
    axes_flat[0].set_ylim(r1_ymin, r1_ymax)
    axes_flat[1].set_ylim(r1_ymin, r1_ymax)
    
    r2_min_data = min(min_medians[2], min_medians[3])
    r2_ymin = np.floor(r2_min_data - 1)
    r2_ymax = max(axes_flat[2].get_ylim()[1], axes_flat[3].get_ylim()[1])
    axes_flat[2].set_ylim(r2_ymin, r2_ymax)
    axes_flat[3].set_ylim(r2_ymin, r2_ymax)

    plt.tight_layout(pad=0.5, w_pad=1.5, h_pad=1.5)
    
    if isinstance(output_files, list):
        for of in output_files:
            plt.savefig(of, bbox_inches='tight', pad_inches=0.02)
        plt.close()
    else:
        plt.savefig(output_files, bbox_inches='tight', pad_inches=0.02)
        plt.close()


def generate_latex_table(df, sadname, output_files):
    # Define method order and display names
    methods = ["GSS", "CWu", "BOP", "BOP-S", "BOP-W", "BOPO", "BOPO-S", "BOPO-W", "Oracle"]
    methods = method_names_short # Added for J2
    if sadname == "OracleSAD":
        label_suffix = "OracleSAD"
    else: # COSAD
        methods = [m for m in methods if m != "GSS" and m != "Oracle"] # Exclude GSS and Oracle for COSAD
        label_suffix = "COSAD"
        
    method_labels = {
        "GSS": r"\gls{GSS}~\cite{horiguchi2021blockonlinegss}",
        "C": r"\gls{C}",
        "CSn": r"\gls{CSn}~\cite{markovich-golan_performance_2018}",
        "CWn": r"\gls{CWn}~\cite{markovich-golan_performance_2018}",
        "CSu": r"\gls{CSu}~\cite{markovich-golan_performance_2018}",
        "CWu": r"\gls{CWu}~\cite{markovich-golan_performance_2018}",
        "BOP": r"\gls{BOP}~\cite{cherkassky_successive_2020}",
        "BOP-S": r"\gls{BOP-S}~\cite{gode2025ebop}",
        "BOP-W": r"\gls{BOP-W}~\cite{gode2025ebop}",
        "BOPO": r"\gls{BOPO}~\cite{gode2025ebop}",
        "BOPO-S": r"\gls{BOPO-S}~\cite{gode2025ebop}",
        "BOPO-W": r"\gls{BOPO-W}~\cite{gode2025ebop}",
        "CB": r"\gls{CB}*",
        "CB-S": r"\gls{CB-S}*",
        "CB-W": r"\gls{CB-W}*",
        "CBW": r"\gls{CBW}~\cite{gode2023covariance}",
        "Oracle": r"Oracle"
    }
    
    # --- Configuration for Precision ---
    # Define decimal places for each metric type
    precision_config = {
        "WMHA": 1,
        "DSINR": 1
    }

    snr_levels = [0, 5, 10, 15]
    
    # Pre-calculate medians
    # Group by input_snr and method, then take median of relevant columns
    cols_of_interest = ["WMHA_A2", "DSINR_A2", "WMHA_A3", "DSINR_A3"]
    medians = df.groupby(["input_snr", "method"])[cols_of_interest].median()
    
    # Helper to find best values for bolding (min for WMHA, max for DSINR)
    # Exclude Oracle from "best" comparison for bolding
    def is_best(val, snr, metric_col, method_name, is_min=True):
        if method_name == "Oracle": 
            return False # Never bold Oracle
            
        current_methods_in_snr = medians.loc[snr].index
        # Filter methods that are in our target list AND not Oracle, GSS (for WMHA)
        comparable_methods = [m for m in methods if m in current_methods_in_snr and m != "Oracle"]
        if metric_col.startswith("WMHA"):
             comparable_methods = [m for m in comparable_methods if m != "GSS"]
             decimals = precision_config["WMHA"]
        else:
             decimals = precision_config["DSINR"]
             
        if not comparable_methods: 
            return False
            
        # Get all relevant values
        vals = [medians.loc[(snr, m)][metric_col] for m in comparable_methods]
        
        # Round the input value and the comparison list
        val_rounded = round(val, decimals)
        vals_rounded = [round(v, decimals) for v in vals]
        
        if is_min:
            best_val = min(vals_rounded)
            # True if the rounded value is less than or equal to the best rounded value
            return val_rounded <= best_val
        else:
            best_val = max(vals_rounded)
            # True if the rounded value is greater than or equal to the best rounded value
            return val_rounded >= best_val
    
    
    # Determine caption based on SAD name
    if sadname == "OracleSAD":
        caption = r"Median weighted Hermitian angle $\psi$ for the conventional and proposed \gls{RTF} vector estimation methods and median \gls{SINR} improvement for all considered methods for different \glspl{SNR} assuming oracle source activity knowledge."
    else:
        caption = r"Median weighted Hermitian angle $\psi$ and \gls{SINR} improvement for the conventional and proposed \gls{RTF} vector estimation methods for different \glspl{SNR} using the online source counting method~\cite{gode2026dnn}."

    # Start constructing LaTeX
    latex_str = r"""\begin{table*}[htbp!]
    \color{blue}
\centering
\begin{tabular}{p{2cm}cccccccccccc}
\toprule
\multirow{3}{2cm}{\\\textbf{Method}} & & \multicolumn{11}{c}{\textbf{\gls{SNR}}} \\
& & \multicolumn{2}{c}{0} & & \multicolumn{2}{c}{5} & & \multicolumn{2}{c}{10} & & \multicolumn{2}{c}{15} \\
\cline{3-4} \cline{6-7} \cline{9-10} \cline{12-13}
 & & $\psi\downarrow$ & $\Delta$\gls{SINR}$\uparrow$ & & $\psi\downarrow$ & $\Delta$\gls{SINR}$\uparrow$ & & $\psi\downarrow$ & $\Delta$\gls{SINR}$\uparrow$ & & $\psi\downarrow$ & $\Delta$\gls{SINR}$\uparrow$ \\
\midrule
\multicolumn{13}{l}{\textbf{2 sources}} \\
\midrule
"""
    
    # --- Part 1: 2 sources (A2) ---
    prec_wmha = precision_config["WMHA"]
    prec_dsinr = precision_config["DSINR"]

    for method in methods:
        row_str = f"{method_labels[method]}         & & "
        for i, snr in enumerate(snr_levels):
            # Check if this method+snr exists in data
            if (snr, method) in medians.index:
                
                # WMHA
                if method == "GSS":
                    wmha_str = "-"
                elif method == "Oracle":
                     # Dynamic formatting for 0.0
                     wmha_str = r"$\SI{" + f"{0.0:.{prec_wmha}f}" + r"}{\degree}$"
                else:
                    val = medians.loc[(snr, method)]["WMHA_A2"]
                    formatted_val = f"{val:.{prec_wmha}f}"
                    if is_best(val, snr, "WMHA_A2", method, is_min=True):
                        wmha_str = r"\textbf{" + f"$\\SI{{{formatted_val}}}{{\\degree}}$" + "}"
                    else:
                        wmha_str = f"$\\SI{{{formatted_val}}}{{\\degree}}$"
                
                # SINR
                val_sinr = medians.loc[(snr, method)]["DSINR_A2"]
                formatted_sinr = f"{val_sinr:.{prec_dsinr}f}"
                if is_best(val_sinr, snr, "DSINR_A2", method, is_min=False):
                    sinr_str = r"\textbf{" + f"$\\SI{{{formatted_sinr}}}{{\\dB}}$" + "}"
                else:
                    sinr_str = f"$\\SI{{{formatted_sinr}}}{{\\dB}}$"
                
                row_str += f"{wmha_str} & {sinr_str}"
            else:
                row_str += " - & - "
                
            if i < len(snr_levels) - 1:
                row_str += " & & "
        
        latex_str += row_str + r" \\" + "\n"

    latex_str += r"""\midrule
\multicolumn{13}{l}{\textbf{3 sources}} \\
\midrule
"""

    # --- Part 2: 3 sources (A3) ---
    for method in methods:
        row_str = f"{method_labels[method]}         & & "
        for i, snr in enumerate(snr_levels):
            if (snr, method) in medians.index:
                 # WMHA
                if method == "GSS":
                    wmha_str = "-"
                elif method == "Oracle":
                     wmha_str = r"$\SI{" + f"{0.0:.{prec_wmha}f}" + r"}{\degree}$"
                else:
                    val = medians.loc[(snr, method)]["WMHA_A3"]
                    formatted_val = f"{val:.{prec_wmha}f}"
                    if is_best(val, snr, "WMHA_A3", method, is_min=True):
                        wmha_str = r"\textbf{" + f"$\\SI{{{formatted_val}}}{{\\degree}}$" + "}"
                    else:
                        wmha_str = f"$\\SI{{{formatted_val}}}{{\\degree}}$"
                
                # SINR
                val_sinr = medians.loc[(snr, method)]["DSINR_A3"]
                formatted_sinr = f"{val_sinr:.{prec_dsinr}f}"
                if is_best(val_sinr, snr, "DSINR_A3", method, is_min=False):
                    sinr_str = r"\textbf{" + f"$\\SI{{{formatted_sinr}}}{{\\dB}}$" + "}"
                else:
                    sinr_str = f"$\\SI{{{formatted_sinr}}}{{\\dB}}$"
                
                row_str += f"{wmha_str} & {sinr_str}"
            else:
                row_str += " - & - "
                
            if i < len(snr_levels) - 1:
                row_str += " & & "
        
        latex_str += row_str + r" \\" + "\n"
        
    # Close table
    latex_str += r"\bottomrule" + "\n"
    latex_str += r"\end{tabular}" + "\n"
    latex_str += r"\vspace{0.1cm}" + "\n"
    latex_str += f"\\caption{{{caption}}}" + "\n"
    latex_str += f"\\label{{tab:combined_{label_suffix}}}" + "\n"
    latex_str += r"\end{table*}" + "\n"
    
    if isinstance(output_files, list):
        for out_file in output_files:
            tex_path = Path(out_file).with_suffix(".tex")
            with open(tex_path, "w") as f:
                f.write(latex_str)
            print(f"Latex table saved to {tex_path}")
    else:
        tex_path = Path(output_files).with_suffix(".tex")
        with open(tex_path, "w") as f:
            f.write(latex_str)
        print(f"Latex table saved to {tex_path}")


# Define the 4 metrics to plot
metrics_to_plot_grid = [
    ("WMHA_A2", r"Weighted Hermitian Angle $\psi_{2}$ [°]", r"RTF Estimation Accuracy (2 sources)"),
    ("WMHA_A3", r"Weighted Hermitian Angle $\psi_{3}$ [°]", r"RTF Estimation Accuracy (3 sources)"),
    ("DSINR_A2", r"SINR Improvement $\Delta\text{SINR}_{2}$ [dB]", r"Source extraction performance (2 sources)"), 
    ("DSINR_A3", r"SINR Improvement $\Delta\text{SINR}_{3}$ [dB]", r"Source extraction performance (3 sources)"),
]

for sadname in ["OracleSAD", "COSAD"]:
    print(f"Generating 2x2 grid plot for {sadname}...")

    # Output path
    filename = f"Main_Exp_WMHA_DSINR_S2_S3_{sadname}.pdf"
    output_path_grid = resultsdirs[sadname] / "plots" / "J2" / filename
    paper_figure_path = Path("/data4/Henri/j3/framewiseSpeakerCounting/documents/Journal2/figures/Plots/revision_caused_by_J1_reviews")
    paper_out_path_grid = paper_figure_path / filename
    
    output_path_grid.parent.mkdir(parents=True, exist_ok=True)
    paper_out_path_grid.parent.mkdir(parents=True, exist_ok=True)

    output_files_list = [output_path_grid, paper_out_path_grid]

    plot_2x2_grid_twocol(
        dfs[sadname].reset_index(drop=True),
        metrics_list=metrics_to_plot_grid,
        group_by="input_snr",
        xlabel_text="Input SNR [dB]",
        output_files=output_files_list,
    )
    print(f"Plot saved to {output_files_list}")
    
    # ALSO GENERATE TABLE with multiple output paths
    generate_latex_table(
        dfs[sadname].reset_index(drop=True),
        sadname,
        output_files_list
    )


Generating 2x2 grid plot for OracleSAD...
Plot saved to [PosixPath('/data4/Henri/j3/framewiseSpeakerCounting/results/J2_RUN/J1_BXLS_main_exp/oracle/plots/J2/Main_Exp_WMHA_DSINR_S2_S3_OracleSAD.pdf'), PosixPath('/data4/Henri/j3/framewiseSpeakerCounting/documents/Journal2/figures/Plots/revision_caused_by_J1_reviews/Main_Exp_WMHA_DSINR_S2_S3_OracleSAD.pdf')]
Latex table saved to /data4/Henri/j3/framewiseSpeakerCounting/results/J2_RUN/J1_BXLS_main_exp/oracle/plots/J2/Main_Exp_WMHA_DSINR_S2_S3_OracleSAD.tex
Latex table saved to /data4/Henri/j3/framewiseSpeakerCounting/documents/Journal2/figures/Plots/revision_caused_by_J1_reviews/Main_Exp_WMHA_DSINR_S2_S3_OracleSAD.tex
Generating 2x2 grid plot for COSAD...
Plot saved to [PosixPath('/data4/Henri/j3/framewiseSpeakerCounting/results/J2_RUN/J1_BXLS_main_exp/PrecomputedSAD/plots/J2/Main_Exp_WMHA_DSINR_S2_S3_COSAD.pdf'), PosixPath('/data4/Henri/j3/framewiseSpeakerCounting/documents/Journal2/figures/Plots/revision_caused_by_J1_reviews/Main_Exp_WMH

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
from pathlib import Path

# 1. defined dimensions from your LaTeX document
latex_two_column_width_pt = 516.0
latex_font_size_pt = 8.0

# 2. Convert to inches for matplotlib
inches_per_pt = 1.0 / 72
fig_width_in = latex_two_column_width_pt * inches_per_pt 
golden_ratio = (5**0.5 - 1) / 2
fig_height_in = fig_width_in * golden_ratio

# 3. Update rcParams
mpl.use("pgf")
mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",
    'font.family': 'serif',
    'text.usetex': True,
    'pgf.rcfonts': False,
    'pgf.preamble': r'\usepackage{amsmath,graphicx,siunitx,bm}',
    'font.size': latex_font_size_pt,
    'axes.labelsize': latex_font_size_pt,      
    'axes.titlesize': latex_font_size_pt,      
    'legend.fontsize': latex_font_size_pt, 
    'xtick.labelsize': latex_font_size_pt, 
    'ytick.labelsize': latex_font_size_pt,
    'figure.figsize': [fig_width_in, fig_height_in],
})

def plot_2x2_grid_twocol(
    df,
    metrics_list, 
    group_by,
    xlabel_text,
    output_files,
):
    if len(metrics_list) > 4:
        print(f"Warning: metrics_list has {len(metrics_list)} entries. Truncating to first 4.")
        metrics_list = metrics_list[:4]
    
    # --- 1. Prepare Consistent Colors & Order ---
    all_methods = [m for m in ["GSS", "CWu", "BOP", "BOP-S", "BOP-W", "BOPO", "BOPO-S", "BOPO-W", "Oracle"]]
    
    palette_colors = sns.color_palette("colorblind", n_colors=len(all_methods))
    method_color_map = dict(zip(all_methods, palette_colors))
    
    # Identify if we are in the COSAD Scenario where GSS should be hidden
    # Use the first output file to check for filename pattern
    if isinstance(output_files, list) and len(output_files) > 0:
        check_path = output_files[0]
    else:
        check_path = output_files
    is_cosad_plot = "COSAD" in str(check_path)

    # Create figure
    fig, axes = plt.subplots(2, 2, figsize=(fig_width_in, fig_height_in), dpi=600)
    axes_flat = axes.flatten()
    letters = ["(a)", "(b)", "(c)", "(d)"]
    min_medians = []

    for i, (metric, ylabel, title) in enumerate(metrics_list):
        ax = axes_flat[i]
        is_bottom = i >= 2
    
        # --- 2. Conditional Filtering ---
        # Exclude GSS if COSAD
        if is_cosad_plot:
            current_df = df[df["method"] != "GSS"]
        else:
            current_df = df

        # Additional Filtering for MHA (Oracle SAD or COSAD doesn't matter, MHA excludes GSS/Oracle)
        if "MHA" in metric:
            plot_df = current_df[~current_df["method"].isin(["GSS", "Oracle"])]
        else:
            plot_df = current_df

        if not plot_df.empty:
            medians = plot_df.groupby([group_by, "method"])[metric].median()
            if not medians.empty:
                min_medians.append(medians.min())
            else:
                min_medians.append(0)
        else:
            min_medians.append(0)

        current_methods = [m for m in all_methods if m in plot_df["method"].unique()]

        sns.barplot(
            data=plot_df,
            x=group_by,
            y=metric,
            hue="method",
            hue_order=current_methods,
            palette=method_color_map,
            errorbar="se",
            estimator=pd.Series.median,
            ax=ax,
            linewidth=0.15,
        )
        
        ax.grid(True, axis='y', linestyle='--', alpha=0.7, linewidth=0.5)
        ax.set_title(title, pad=3) 
        ax.set_ylabel(ylabel)
        
        if is_bottom:
            ax.set_xlabel(xlabel_text)
        else:
            ax.set_xlabel("")
            ax.set_xticklabels([]) 
            
        ax.text(-0.06, 1.1, r"{" + letters[i] + "}", 
                transform=ax.transAxes, va='top', ha='right', fontsize=latex_font_size_pt)

        if ax.get_legend():
            ax.get_legend().remove()

        # Legend Logic (Index 0)
        if i == 0:
            legend_handles = []
            
            # Iterate through ALL methods to maintain grid positions
            for m in all_methods:
                if is_cosad_plot and m in ["GSS", "Oracle"]:
                    # Create an invisible patch for GSS (preserves the grid slot)
                    # Label is empty or spaces to keep alignment without showing text
                    legend_handles.append(mpatches.Patch(color="none", label="", alpha=0))
                else:
                    legend_handles.append(mpatches.Patch(color=method_color_map[m], label=m))
            
            ax.legend(
                handles=legend_handles,
                loc='upper right', 
                ncol=3, 
                frameon=True,
                columnspacing=0.5,
                handletextpad=0.2,
                labelspacing=0.2,
                fontsize=latex_font_size_pt
            )

    # --- Shared Y-Axis Scaling ---
    r1_min_data = min(min_medians[0], min_medians[1])
    r1_ymin = np.floor(r1_min_data - 1)
    r1_ymax = max(axes_flat[0].get_ylim()[1], axes_flat[1].get_ylim()[1])
    axes_flat[0].set_ylim(r1_ymin, r1_ymax)
    axes_flat[1].set_ylim(r1_ymin, r1_ymax)
    
    r2_min_data = min(min_medians[2], min_medians[3])
    r2_ymin = np.floor(r2_min_data - 1)
    r2_ymax = max(axes_flat[2].get_ylim()[1], axes_flat[3].get_ylim()[1])
    
    if True:
        r2_ymin = 0.4
        r2_ymax = 0.9
    
    axes_flat[2].set_ylim(r2_ymin, r2_ymax)
    axes_flat[3].set_ylim(r2_ymin, r2_ymax)

    plt.tight_layout(pad=0.5, w_pad=1.5, h_pad=1.5)
    
    if isinstance(output_files, list):
        for of in output_files:
            plt.savefig(of, bbox_inches='tight', pad_inches=0.02)
        plt.close()
    else:
        plt.savefig(output_files, bbox_inches='tight', pad_inches=0.02)
        plt.close()


def generate_latex_table(df, sadname, metrics, output_files):
    # Define method order and display names
    if sadname == "OracleSAD":
        methods = ["GSS", "CWu", "BOP", "BOP-S", "BOP-W", "BOPO", "BOPO-S", "BOPO-W", "Oracle"]
        label_suffix = "OracleSAD"
    else: # COSAD
        methods = ["CWu", "BOP", "BOP-S", "BOP-W", "BOPO", "BOPO-S", "BOPO-W"]
        label_suffix = "COSAD"
        
    method_labels = {
        "GSS": r"\gls{GSS}~\cite{horiguchi2021blockonlinegss}",
        "CWu": r"\gls{CWu}~\cite{markovich-golan_performance_2018}",
        "BOP": r"\gls{BOP}~\cite{cherkassky_successive_2020}",
        "BOP-S": r"\gls{BOP-S}*",
        "BOP-W": r"\gls{BOP-W}*",
        "BOPO": r"\gls{BOPO}*",
        "BOPO-S": r"\gls{BOPO-S}*",
        "BOPO-W": r"\gls{BOPO-W}*",
        "Oracle": r"Oracle"
    }

    snr_levels = [0, 5, 10, 15]
    
    # Pre-calculate medians
    # Group by input_snr and method, then take median of relevant columns
    # cols_of_interest = ["WMHA_A2", "DSINR_A2", "WMHA_A3", "DSINR_A3"]
    cols_of_interest = metrics
    medians = df.groupby(["input_snr", "method"])[cols_of_interest].median()
    
    # Helper to find best values for bolding (min for WMHA, max for DSINR)
    # Exclude Oracle from "best" comparison for bolding
    def is_best(val, snr, metric_col, method_name, is_min=True):
        if method_name == "Oracle": 
            return False # Never bold Oracle
            
        current_methods_in_snr = medians.loc[snr].index
        # Filter methods that are in our target list AND not Oracle, GSS (for WMHA)
        comparable_methods = [m for m in methods if m in current_methods_in_snr and m != "Oracle"]
        if metric_col.startswith("WMHA"):
             comparable_methods = [m for m in comparable_methods if m != "GSS"]
             
        if not comparable_methods: 
            return False
            
        vals = [medians.loc[(snr, m)][metric_col] for m in comparable_methods]
        
        if is_min:
            best_val = min(vals)
            return val <= best_val + 1e-6 # float tolerance
        else:
            best_val = max(vals)
            return val >= best_val - 1e-6
    
    
    # Determine caption based on SAD name
    if sadname == "OracleSAD":
        caption = r"Median weighted Hermitian angle $\psi$ for the conventional and proposed \gls{RTF} vector estimation methods and median \gls{SINR} improvement for all considered methods for different \glspl{SNR} assuming oracle source activity knowledge."
    else:
        caption = r"Median weighted Hermitian angle $\psi$ for the conventional and proposed \gls{RTF} vector estimation methods and median \gls{SINR} improvement for all considered methods for different \glspl{SNR} using the online source counting method~\cite{gode2026dnn}."

    # Start constructing LaTeX
    latex_str = r"""\begin{table*}[htbp]
    \color{blue}
\centering
\begin{tabular}{p{2cm}cccccccccccc}
\toprule
\multirow{3}{2cm}{\\\textbf{Method}} & & \multicolumn{11}{c}{\textbf{\gls{SNR}}} \\
& & \multicolumn{2}{c}{0} & & \multicolumn{2}{c}{5} & & \multicolumn{2}{c}{10} & & \multicolumn{2}{c}{15} \\
\cline{3-4} \cline{6-7} \cline{9-10} \cline{12-13}
 & & $\psi\downarrow$ & $\Delta$\gls{SINR}$\uparrow$ & & $\psi\downarrow$ & $\Delta$\gls{SINR}$\uparrow$ & & $\psi\downarrow$ & $\Delta$\gls{SINR}$\uparrow$ & & $\psi\downarrow$ & $\Delta$\gls{SINR}$\uparrow$ \\
\midrule
\multicolumn{13}{l}{\textbf{2 sources}} \\
\midrule
"""
    
    # --- Part 1: 2 sources (A2) ---
    for method in methods:
        row_str = f"{method_labels[method]}         & & "
        for i, snr in enumerate(snr_levels):
            # Check if this method+snr exists in data
            if (snr, method) in medians.index:
                
                # WMHA
                if method == "GSS":
                    wmha_str = "-"
                elif method == "Oracle":
                     wmha_str = r"$\SI{0.0}{\degree}$"
                else:
                    val = medians.loc[(snr, method)]["WMHA_A2"]
                    if is_best(val, snr, "WMHA_A2", method, is_min=True):
                        wmha_str = r"\textbf{" + f"$\\SI{{{val:.1f}}}{{\\degree}}$" + "}"
                    else:
                        wmha_str = f"$\\SI{{{val:.1f}}}{{\\degree}}$"
                
                # SINR
                val_sinr = medians.loc[(snr, method)]["DSINR_A2"]
                if is_best(val_sinr, snr, "DSINR_A2", method, is_min=False):
                    sinr_str = r"\textbf{" + f"$\\SI{{{val_sinr:.1f}}}{{\\dB}}$" + "}"
                else:
                    sinr_str = f"$\\SI{{{val_sinr:.1f}}}{{\\dB}}$"
                
                row_str += f"{wmha_str} & {sinr_str}"
            else:
                row_str += " - & - "
                
            if i < len(snr_levels) - 1:
                row_str += " & & "
        
        latex_str += row_str + r" \\" + "\n"

    latex_str += r"""\midrule
\multicolumn{13}{l}{\textbf{3 sources}} \\
\midrule
"""

    # --- Part 2: 3 sources (A3) ---
    for method in methods:
        row_str = f"{method_labels[method]}         & & "
        for i, snr in enumerate(snr_levels):
            if (snr, method) in medians.index:
                 # WMHA
                if method == "GSS":
                    wmha_str = "-"
                elif method == "Oracle":
                     wmha_str = r"$\SI{0.0}{\degree}$"
                else:
                    val = medians.loc[(snr, method)]["WMHA_A3"]
                    if is_best(val, snr, "WMHA_A3", method, is_min=True):
                        wmha_str = r"\textbf{" + f"$\\SI{{{val:.1f}}}{{\\degree}}$" + "}"
                    else:
                        wmha_str = f"$\\SI{{{val:.1f}}}{{\\degree}}$"
                
                # SINR
                val_sinr = medians.loc[(snr, method)]["DSINR_A3"]
                if is_best(val_sinr, snr, "DSINR_A3", method, is_min=False):
                    sinr_str = r"\textbf{" + f"$\\SI{{{val_sinr:.1f}}}{{\\dB}}$" + "}"
                else:
                    sinr_str = f"$\\SI{{{val_sinr:.1f}}}{{\\dB}}$"
                
                row_str += f"{wmha_str} & {sinr_str}"
            else:
                row_str += " - & - "
                
            if i < len(snr_levels) - 1:
                row_str += " & & "
        
        latex_str += row_str + r" \\" + "\n"
        
    # Close table
    latex_str += r"\bottomrule" + "\n"
    latex_str += r"\end{tabular}" + "\n"
    latex_str += r"\vspace{0.1cm}" + "\n"
    latex_str += f"\\caption{{{caption}}}" + "\n"
    latex_str += f"\\label{{tab:combined_{label_suffix}}}" + "\n"
    latex_str += r"\end{table*}" + "\n"
    
    if isinstance(output_files, list):
        for out_file in output_files:
            tex_path = Path(out_file).with_suffix(".tex")
            with open(tex_path, "w") as f:
                f.write(latex_str)
            print(f"Latex table saved to {tex_path}")
    else:
        tex_path = Path(output_files).with_suffix(".tex")
        with open(tex_path, "w") as f:
            f.write(latex_str)
        print(f"Latex table saved to {tex_path}")


# Define the 4 metrics to plot
metrics_to_plot_grid = [
    ("WMHA_A2", r"Weighted Hermitian Angle $\psi_{2}$ [°]", r"RTF Estimation Accuracy (2 sources)"),
    ("WMHA_A3", r"Weighted Hermitian Angle $\psi_{3}$ [°]", r"RTF Estimation Accuracy (3 sources)"),
    ("DSINR_A2", r"SINR Improvement [dB]", r"Source extraction performance (2 sources)"), 
    ("DSINR_A3", r"SINR Improvement [dB]", r"Source extraction performance (3 sources)"),
    # ("STOIo_A2", r"STOI", r"Source extraction performance (2 sources)"), 
    # ("STOIo_A3", r"STOI", r"Source extraction performance (3 sources)"),
]
    
for sadname in ["OracleSAD", "COSAD"]:
    print(f"Generating 2x2 grid plot for {sadname}...")

    # Output path
    output_path_grid = resultsdirs[sadname] / "plots" / f"Main_Exp_WMHA_DSINR_S2_S3_{sadname}.pdf"
    paper_figure_path = Path("/data4/Henri/j3/framewiseSpeakerCounting/documents/Journal1/figures/Rebuttal")
    paper_out_path_grid = paper_figure_path / f"Main_Exp_WMHA_DSINR_S2_S3_{sadname}.pdf"
    
    output_path_grid.parent.mkdir(parents=True, exist_ok=True)
    paper_out_path_grid.parent.mkdir(parents=True, exist_ok=True)

    output_files_list = [output_path_grid, paper_out_path_grid]

    plot_2x2_grid_twocol(
        dfs[sadname].reset_index(drop=True),
        metrics_list=metrics_to_plot_grid,
        group_by="input_snr",
        xlabel_text="Input SNR [dB]",
        output_files=output_files_list,
    )
    print(f"Plot saved to {output_files_list}")
    
    
    # ALSO GENERATE TABLE with multiple output paths
    generate_latex_table(
        dfs[sadname].reset_index(drop=True),
        sadname,
        [met for met, _, _ in metrics_to_plot_grid],
        output_files_list
    )


Generating 2x2 grid plot for OracleSAD...
Plot saved to [PosixPath('/data4/Henri/j3/framewiseSpeakerCounting/results/J1_BXLS_main_exp_OracleSAD/plots/Main_Exp_WMHA_STOIo_S2_S3_OracleSAD.pdf'), PosixPath('/data4/Henri/j3/framewiseSpeakerCounting/documents/Journal1/figures/Rebuttal/Main_Exp_WMHA_STOIo_S2_S3_OracleSAD.pdf')]
Latex table saved to /data4/Henri/j3/framewiseSpeakerCounting/results/J1_BXLS_main_exp_OracleSAD/plots/Main_Exp_WMHA_STOIo_S2_S3_OracleSAD.tex
Latex table saved to /data4/Henri/j3/framewiseSpeakerCounting/documents/Journal1/figures/Rebuttal/Main_Exp_WMHA_STOIo_S2_S3_OracleSAD.tex
Generating 2x2 grid plot for COSAD...
Plot saved to [PosixPath('/data4/Henri/j3/framewiseSpeakerCounting/results/J1_BXLS_main_exp_COSAD/plots/Main_Exp_WMHA_STOIo_S2_S3_COSAD.pdf'), PosixPath('/data4/Henri/j3/framewiseSpeakerCounting/documents/Journal1/figures/Rebuttal/Main_Exp_WMHA_STOIo_S2_S3_COSAD.pdf')]
Latex table saved to /data4/Henri/j3/framewiseSpeakerCounting/results/J1_BXLS_main_exp_

In [ ]:
# I want to have a torch tensor of size [2, 2, 4, 4, 9] whereby the dimensions correspond to the following:
# [metrics, sad_method, segment, input_snr, methods]
# metrics are either WMHA or DSINR
# sad_method is either OracleSAD or COSAD
# segment is either "" (empty referring to utterance level, A1, A2 or A3
# input_snr is either 0, 5, 10 or 15
# methods are the method_names_short defined above
import torch
import numpy as np

# 1. Define dimensions
metrics = ["WMHA", "DSINR"]
sad_methods = ["OracleSAD", "COSAD"]
# Map segment conceptual names to possible column suffixes
segment_suffixes = ["", "_A1", "_A2", "_A3"] 
input_snrs = [0, 5, 10, 15]

# Use the 9 methods corresponding to the "9" in your requested dimension
# (Excluding GSSL which was in the original loading list but not in the tables)
methods_tensor_order = ["GSS", "CWu", "BOP", "BOP-S", "BOP-W", "BOPO", "BOPO-S", "BOPO-W", "Oracle"]

# 2. Initialize Tensor
results_tensor = torch.zeros(
    len(metrics), 
    len(sad_methods), 
    len(segment_suffixes), 
    len(input_snrs), 
    len(methods_tensor_order)
)

# 3. Fill Tensor
for i_met, metric in enumerate(metrics):
    for i_sad, sad in enumerate(sad_methods):
        if sad not in dfs:
            print(f"Skipping {sad}, not in dfs")
            continue
            
        df = dfs[sad]
        
        for i_seg, seg_suffix in enumerate(segment_suffixes):
            col_name = f"{metric}{seg_suffix}"
            
            # Helper to check if column exists (e.g. A1 might be missing)
            if col_name not in df.columns:
                # print(f"  Note: Column {col_name} not found in {sad} dataframe. Filling with NaN.")
                results_tensor[i_met, i_sad, i_seg, :, :] = float('nan')
                continue
            
            for i_snr, snr in enumerate(input_snrs):
                for i_meth, method in enumerate(methods_tensor_order):
                    # Filter mask
                    mask = (df["input_snr"] == snr) & (df["method"] == method)
                    values = df.loc[mask, col_name]
                    
                    if not values.empty:
                        results_tensor[i_met, i_sad, i_seg, i_snr, i_meth] = values.median()
                    else:
                        results_tensor[i_met, i_sad, i_seg, i_snr, i_meth] = float('nan')

print(f"Created tensor with shape: {results_tensor.shape}")
print("Dimensions: [metrics, sad_method, segment, input_snr, methods]")

Created tensor with shape: torch.Size([2, 2, 4, 4, 9])
Dimensions: [metrics, sad_method, segment, input_snr, methods]


In [6]:
compare_tensor = torch.round(results_tensor[1, 1] - results_tensor[1, 0], decimals=2)
# print the values of the compare tensor in a nicely readable format (maybe as a table)

# Dimensions of compare_tensor are [segment, input_snr, methods]
# segment indices: 0="", 1="_A1", 2="_A2", 3="_A3"
segment_names = ["Utterance", "Segment A1", "Segment A2", "Segment A3"]

# Use pandas for nice formatting
import pandas as pd
from IPython.display import display

for i_seg, seg_name in enumerate(segment_names):
    print(f"\n=======================================================")
    print(f"DSINR Difference (COSAD - OracleSAD) for {seg_name}")
    print(f"=======================================================")
    
    # Extract the 2D slice for this segment: [input_snr, methods]
    data_slice = compare_tensor[i_seg].numpy()
    
    # Create DataFrame for display
    # shape is (4, 9) -> rows=SNRs, cols=Methods
    df_compare = pd.DataFrame(
        data_slice, 
        index=[f"SNR {s}" for s in input_snrs], 
        columns=methods_tensor_order
    )
    
    display(df_compare)


DSINR Difference (COSAD - OracleSAD) for Utterance


,GSS,CWv,BOP,BOP-S,BOP-W,BOPO,BOPO-S,BOPO-W,Oracle
SNR 0,0.0,-0.47,-0.46,-0.51,-0.61,-0.51,-0.58,-0.74,NaN
SNR 5,0.0,-0.40,-0.53,-0.53,-0.61,-0.62,-0.61,-0.70,NaN
SNR 10,0.0,-0.49,-0.49,-0.41,-0.32,-0.49,-0.62,-0.48,NaN
SNR 15,0.0,-0.45,-0.44,-0.44,-0.34,-0.61,-0.59,-0.52,NaN



DSINR Difference (COSAD - OracleSAD) for Segment A1


,GSS,CWv,BOP,BOP-S,BOP-W,BOPO,BOPO-S,BOPO-W,Oracle
SNR 0,0.0,0.13,0.13,0.12,0.12,0.19,0.20,0.14,NaN
SNR 5,0.0,0.14,0.15,0.17,0.12,0.14,0.22,0.12,NaN
SNR 10,0.0,0.07,0.16,0.06,0.02,0.13,0.08,0.08,NaN
SNR 15,0.0,0.09,0.05,0.07,-0.02,0.03,0.06,0.08,NaN



DSINR Difference (COSAD - OracleSAD) for Segment A2


,GSS,CWv,BOP,BOP-S,BOP-W,BOPO,BOPO-S,BOPO-W,Oracle
SNR 0,0.0,-0.66,-0.31,-0.49,-0.64,-0.47,-0.52,-0.78,NaN
SNR 5,0.0,-0.52,-0.40,-0.52,-0.58,-0.57,-0.60,-0.76,NaN
SNR 10,0.0,-0.50,-0.33,-0.52,-0.39,-0.58,-0.60,-0.60,NaN
SNR 15,0.0,-0.46,-0.29,-0.27,-0.39,-0.42,-0.46,-0.45,NaN



DSINR Difference (COSAD - OracleSAD) for Segment A3


,GSS,CWv,BOP,BOP-S,BOP-W,BOPO,BOPO-S,BOPO-W,Oracle
SNR 0,0.0,-0.46,-0.64,-0.64,-0.58,-0.71,-0.66,-0.76,NaN
SNR 5,0.0,-0.40,-0.52,-0.58,-0.61,-0.64,-0.59,-0.54,NaN
SNR 10,0.0,-0.35,-0.48,-0.53,-0.48,-0.53,-0.55,-0.58,NaN
SNR 15,0.0,-0.37,-0.44,-0.46,-0.46,-0.48,-0.53,-0.56,NaN


In [ ]:
compare_tensor = torch.round(results_tensor[1, 1] - results_tensor[1, 0], decimals=2)
# print the values of the compare tensor in a nicely readable format (maybe as a table)




In [ ]:
# # Use seaborn to create a barplot showing the median of the selected metric
# # with its std error of the median as error bars for each method,
# # grouped by a another metric (usually a meta info like input SNR or segment length)
# import seaborn as sns
# import matplotlib.pyplot as plt
# def plot_metric_by_group(
#     df,
#     metric,
#     group_by,
#     xlabel,
#     title,
#     ylabel,
#     output_file,
# ):
#     # 1. Determine order and calculate counts
#     # Get unique values and sort them to ensure consistent ordering
#     groups = sorted(df[group_by].unique())
#     # Calculate number of rows for each group value
#     counts = df[group_by].value_counts()
    
#     plt.figure(figsize=(10, 6))
    
#     # 2. Pass 'order' to ensure the plot follows our sorted groups
#     ax = sns.barplot(
#         data=df,
#         x=group_by,
#         y=metric,
#         hue="method",
#         errorbar="se",
#         estimator=pd.Series.median,
#         capsize=0.1,
#         order=groups, 
#     )
    
#     # 3. Create new labels with counts
#     # Example label: "0.5\n(N=1200)"
#     new_labels = [f"{g}\n({counts[g]})" for g in groups]
#     ax.set_xticklabels(new_labels)

#     ax.grid(True, axis='y', linestyle='--', alpha=0.7)
#     ax.set_title(title)
#     ax.set_ylabel(ylabel)
#     ax.set_xlabel(xlabel)
#     plt.legend(title="Method", bbox_to_anchor=(1.05, 1), loc='upper left')
#     plt.tight_layout()
#     plt.savefig(output_file)
#     plt.close()
    
# # Now create plots for different metrics
# metrics_to_plot = [
#     ("WMHA_A2", "WMHA [°]", "Segment A2"),
#     ("WMHA_A3", "WMHA [°]", "Segment A3"),
#     ("DSINR_A2", "SINR [dB]", "Segment A2"),
#     ("DSINR_A3", "SINR [dB]", "Segment A3"),
#     ("STOIo_A2", "STOI", "Segment A2"),
#     ("STOIo_A3", "STOI", "Segment A3"),
# ]
# for metric, ylabel, title in metrics_to_plot:
#     plot_metric_by_group(
#         df_all,
#         metric=metric,
#         group_by="input_snr",
#         xlabel="Input SNR [dB]",
#         title=f"{title}",
#         ylabel=ylabel,
#         output_file=resultsdir / "plots" /f"{metric}_by_input_snr.png",
#     )

/tmp/ipykernel_2164341/2380907857.py:38: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(new_labels)


/tmp/ipykernel_2164341/2380907857.py:38: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(new_labels)
/tmp/ipykernel_2164341/2380907857.py:38: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(new_labels)
/tmp/ipykernel_2164341/2380907857.py:38: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(new_labels)
/tmp/ipykernel_2164341/2380907857.py:38: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(new_labels)
/tmp/ipykernel_2164341/2380907857.py:38: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ne

: 

: 

: 

In [28]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
from pathlib import Path
import string

# ==============================================================================
# 1. CORE FUNCTIONS (Generic Logic)
# ==============================================================================

def setup_matplotlib_rc(n_metrics, n_segments, latex_two_column_width_pt=516.0, latex_font_size_pt=8.0):
    inches_per_pt = 1.0 / 72
    fig_width_in = latex_two_column_width_pt * inches_per_pt * 2
    golden_ratio = (5**0.5 - 1) / 2
    
    # Each individual subplot should have a golden ratio.
    # Therefore the overall height ratio is height_ratio * (rows / cols)
    fig_height_in = fig_width_in * golden_ratio * (n_metrics / n_segments)

    mpl.use("pgf")
    mpl.rcParams.update({
        "pgf.texsystem": "pdflatex",
        'font.family': 'serif',
        'text.usetex': True,
        'pgf.rcfonts': False,
        'pgf.preamble': r'\usepackage{amsmath,graphicx,siunitx,bm}',
        'font.size': latex_font_size_pt,
        'axes.labelsize': latex_font_size_pt,      
        'axes.titlesize': latex_font_size_pt,      
        'legend.fontsize': latex_font_size_pt, 
        'xtick.labelsize': latex_font_size_pt, 
        'ytick.labelsize': latex_font_size_pt,
        'figure.figsize': [fig_width_in, fig_height_in],
    })
    return fig_width_in, fig_height_in

def plot_generic_grid(df, metrics_config, segments_config, group_by, xlabel_text, output_files, all_methods, is_cosad, fig_dims, force_y_limits=None):
    fig_width_in, fig_height_in = fig_dims
    n_rows = len(metrics_config)
    n_cols = len(segments_config)
    
    # Setup Colors
    palette_colors = sns.color_palette("colorblind", n_colors=len(all_methods))
    method_color_map = dict(zip(all_methods, palette_colors))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_width_in, fig_height_in), dpi=600, squeeze=False)
    letters = list(string.ascii_lowercase)
    
    # Store min/max limits for each row to synchronize y-axes
    row_limits = {r: [float('inf'), float('-inf')] for r in range(n_rows)}

    plot_idx = 0
    for r, metric_info in enumerate(metrics_config):
        for c, segment_info in enumerate(segments_config):
            ax = axes[r, c]
            segment_str = "_" + segment_info['id'] if segment_info['id'] != "" else ""
            bfstr = metric_info.get('BF', None)
            if bfstr is not None:
                metric_col = f"{metric_info['name']}_{bfstr}{segment_str}"
            else:
                metric_col = f"{metric_info['name']}{segment_str}"
            is_bottom = (r == n_rows - 1)
            
            # --- Filtering ---
            plot_df = df.copy()
            if is_cosad: plot_df = plot_df[plot_df["method"] != "GSS"]
            if metric_info.get('exclude_oracle_gss', False): 
                plot_df = plot_df[~plot_df["method"].isin(["GSS", "Oracle"])]
                
            if plot_df.empty or metric_col not in plot_df.columns:
                ax.set_visible(False)
                continue

            current_methods = [m for m in all_methods if m in plot_df["method"].unique()]

            # --- Plotting ---
            sns.barplot(
                data=plot_df, x=group_by, y=metric_col,
                hue="method", hue_order=current_methods, palette=method_color_map,
                errorbar="se", estimator=pd.Series.median, ax=ax, linewidth=0.15,
            )
            
            # Record limits for this row
            medians = plot_df[plot_df['method'].isin(current_methods)].groupby([group_by, "method"])[metric_col].median()
            if not medians.empty:
                row_limits[r][0] = min(row_limits[r][0], medians.min())
            # row_limits[r][0] = min(row_limits[r][0], ax.get_ylim()[0])    
            row_limits[r][1] = max(row_limits[r][1], ax.get_ylim()[1])
            
            # --- Styling ---
            ax.grid(True, axis='y', linestyle='--', alpha=0.7, linewidth=0.5)
            ax.set_title(f"{metric_info['title']} ({segment_info['title']})", pad=3) 
            ax.set_ylabel(metric_info['ylabel'])
            
            if is_bottom: ax.set_xlabel(xlabel_text)
            else: ax.set_xlabel(""); ax.set_xticklabels([]) 
                
            ax.text(-0.06, 1.1, r"{(" + letters[plot_idx] + ")}", transform=ax.transAxes, va='top', ha='right')
            plot_idx += 1

            if ax.get_legend(): ax.get_legend().remove()

            # --- Master Legend (Top Left Plot) ---
            if r == 0 and c == 0:
                handles = []
                for m in all_methods:
                    if is_cosad and m in ["GSS", "Oracle"]:
                        handles.append(mpatches.Patch(color="none", label="", alpha=0))
                    else:
                        handles.append(mpatches.Patch(color=method_color_map[m], label=m))
                ax.legend(handles=handles, loc='upper right', ncol=3, frameon=True, 
                          columnspacing=0.5, handletextpad=0.2, labelspacing=0.2)

    # --- Synchronize Y-Axes per row ---
    for r, metric_info in enumerate(metrics_config):
        # Allow precision to be either a float (e.g., 0.1, 0.01) or an int representing decimal places
        precision_val = metric_info.get('precision', 1)
        
        if isinstance(precision_val, float):
            precision_step = precision_val
            # Calculate format string precision based on step
            fmt_dec = len(str(precision_step).split('.')[-1]) if '.' in str(precision_step) else 0
        else:
            # Backwards compatibility: if precision is int (e.g. 1 means 0.1 step)
            precision_step = 10**(-precision_val)
            fmt_dec = precision_val
            
        # Floor based on precision step instead of just integer floor
        if row_limits[r][0] != float('inf'):
            # Multiply by 1/step, floor, then convert back
            factor = 1.0 / precision_step
            y_min = np.floor(row_limits[r][0] * factor) / factor #- precision_step
        else:
            y_min = 0
            
        y_max = row_limits[r][1]
        
        # Apply manual override if provided in config
        if force_y_limits and metric_info['name'] in force_y_limits:
            y_min, y_max = force_y_limits[metric_info['name']]

        for c in range(n_cols):
            axes[r, c].set_ylim(y_min, y_max)
            
            # Optionally format y-ticks according to precision
            axes[r, c].yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter(f'%.{fmt_dec}f'))

    plt.tight_layout(pad=0.5, w_pad=1.5, h_pad=1.5)
    
    # Save
    if not isinstance(output_files, list): output_files = [output_files]
    for of in output_files:
        plt.savefig(of, bbox_inches='tight', pad_inches=0.02)
    plt.close()


def generate_generic_latex_table(df, sadname, metrics_config, segments_config, all_methods, method_labels, snr_levels, output_files, caption):
    is_cosad = (sadname == "COSAD")
    methods = [m for m in all_methods if not (is_cosad and m in ["GSS", "Oracle"])]
    
    # Latex Header Construction
    num_cols = len(snr_levels) * len(metrics_config) + 2
    header_align = "p{2cm}" + "c" * (num_cols - 1)
    
    latex_str = f"\\begin{{table*}}[htbp!]\n\\color{{blue}}\n\\centering\n\\begin{{tabular}}{{{header_align}}}\n\\toprule\n"
    latex_str += "\\multirow{3}{2cm}{\\\\\\textbf{Method}} & & \\multicolumn{" + str(num_cols - 2) + "}{c}{\\textbf{\\gls{SNR}}} \\\\\n"
    
    # SNR Headers
    snr_row = "&"
    for snr in snr_levels:
        snr_row += f" & \\multicolumn{{{len(metrics_config)}}}{{c}}{{{snr}}} &"
    latex_str += snr_row[:-1] + "\\\\\n"
    
    # Line rules
    line_start = 3
    for i in range(len(snr_levels)):
        latex_str += f"\\cline{{{line_start}-{line_start+len(metrics_config)-1}}} "
        line_start += len(metrics_config) + 1
    latex_str += "\n"
    
    # Metric Headers
    metric_headers = "& & " + " & ".join([m['latex_header'] for m in metrics_config])
    latex_str += metric_headers * len(snr_levels) + " \\\\\n\\midrule\n"
    
    # --- Generate Rows per Segment ---
    for seg in segments_config:
        latex_str += f"\\multicolumn{{{num_cols}}}{{l}}{{\\textbf{{{seg['title']}}}}} \\\\\n\\midrule\n"
        
        # Pre-calc medians for this segment
        segment_str = "_" + seg['id'] if seg['id'] != "" else ""
        cols_of_interest = [f"{m['name']}{segment_str}" for m in metrics_config]
        available_cols = [c for c in cols_of_interest if c in df.columns]
        if not available_cols: continue
        medians = df.groupby(["input_snr", "method"])[available_cols].median()
        
        for method in methods:
            row_str = f"{method_labels.get(method, method)} & & "
            for i, snr in enumerate(snr_levels):
                if (snr, method) in medians.index:
                    for m_idx, metric in enumerate(metrics_config):
                        bf_str = metric.get('BF', None)
                        if bf_str is not None:
                            col_name = f"{metric['name']}_{bf_str}{segment_str}"
                        else:
                            col_name = f"{metric['name']}{segment_str}"
                        is_best_min = metric.get('is_min_better', True)
                        dec = metric.get('precision', 1)
                        unit = metric.get('unit', '')
                        
                        # Handle special cases (Oracle, GSS)
                        if method == "GSS" and metric.get('exclude_oracle_gss'): val_str = "-"
                        elif method == "Oracle" and metric.get('exclude_oracle_gss'): val_str = f"$\\SI{{0.0}}{{ {unit} }}$"
                        else:
                            val = medians.loc[(snr, method), col_name]
                            if pd.isna(val): val_str = "-"
                            else:
                                # Best finding logic
                                comp_methods = [m for m in methods if m != "Oracle"]
                                if metric.get('exclude_oracle_gss'): comp_methods = [m for m in comp_methods if m != "GSS"]
                                
                                is_best = False
                                if comp_methods and method != "Oracle":
                                    vals = medians.loc[snr, col_name].reindex(comp_methods).dropna()
                                    if not vals.empty:
                                        best_v = vals.min() if is_best_min else vals.max()
                                        is_best = (val <= best_v + 1e-6) if is_best_min else (val >= best_v - 1e-6)
                                
                                # Format value based on precision type
                                dec_val = metric.get('precision', 1)
                                if isinstance(dec_val, float):
                                    fmt_dec = len(str(dec_val).split('.')[-1]) if '.' in str(dec_val) else 0
                                else:
                                    fmt_dec = dec_val
                                    
                                fmt_val = f"{val:.{fmt_dec}f}"
                                if unit: val_str_base = f"$\\SI{{{fmt_val}}}{{{unit}}}$"
                                else: val_str_base = fmt_val
                                
                                val_str = f"\\textbf{{{val_str_base}}}" if is_best else val_str_base
                                
                        row_str += val_str + (" & " if m_idx < len(metrics_config)-1 else "")
                else:
                    row_str += " - & " * (len(metrics_config)-1) + " - "
                    
                if i < len(snr_levels) - 1: row_str += " & & "
            latex_str += row_str + r" \\" + "\n"
            
    # Finalize Table
    latex_str += r"\bottomrule" + "\n\\end{tabular}\n\\vspace{0.1cm}\n"
    latex_str += f"\\caption{{{caption}}}\n\\label{{tab:combined_{sadname}}}\n\\end{{table*}}\n"
    
    # Save
    if not isinstance(output_files, list): output_files = [output_files]
    for out_file in output_files:
        tex_path = Path(out_file).with_suffix(".tex")
        with open(tex_path, "w") as f: f.write(latex_str)


# ==============================================================================
# 2. USER CONFIGURATION BLOCK
# ==============================================================================

# Data
# Expected variables from above cells: dfs (dictionary of dataframes), method_names_short
EXPERIMENTS = ["OracleSAD", "COSAD"]
ALL_METHODS = [
  # "online-cgmm-mvdr",
#   "GSS",
#   "C",
#   "CSn",
#   "CWn",
#   "CSu",
# #   "CWu",
# #   "CBW",
# #   "BOP", 
#   "BOP-S", 
#   "BOP-W", 
#   "BOPO", 
#   "BOPO-S", 
  "BOPO-W",
# #   "CB",
# #   "CB-S",
  "CB-W",
  "Oracle", 
]
GROUP_BY = "input_snr_5dBsteps"
XLABEL = "Input SNR [dB]"
SNR_LEVELS = [0, 5, 10, 15]



# -----------------
# Define Plot Columns (Segments)
# -----------------
SEGMENTS = [
    {"id": "A1", "title": "1 source"},
    {"id": "A2", "title": "2 sources"},
    {"id": "A3", "title": "3 sources"},
    # {"id": "D1", "title": "1 source deact"},
    # {"id": "D2", "title": "2 sources deact"},    
    {"id": "", "title": "all"},
]

# -----------------
# Define Plot Rows (Metrics for Tables & Figures)
# -----------------
BF = 'LCMV'
METRICS = [
    {
        "name": "WMHA", 
        "ylabel": r"Weighted Hermitian Angle $\psi$ [°]", 
        "title": "RTF Estimation Accuracy",
        "latex_header": r"$\psi\downarrow$",
        "unit": r"\degree",
        "precision": 0,
        "is_min_better": True,
        "exclude_oracle_gss": True # Specific to WMHA logic
    },
    {
        "name": "DSINR", 
        "ylabel": r"SINR Improvement $\Delta\text{SINR}$ [dB]", 
        "title": "Source Extraction Performance",
        "latex_header": r"$\Delta$\gls{SINR}$\uparrow$",
        "unit": r"\dB",
        "precision": 0,
        "is_min_better": False,
        "exclude_oracle_gss": False,
        "BF": BF,
    },
    {
        "name": "PESQo", 
        "ylabel": r"$\text{PESQ}$", 
        "title": "Source Extraction Performance",
        "latex_header": r"$\gls{PESQ}$",
        "unit": r"",
        "precision": 1,
        "is_min_better": False,
        "exclude_oracle_gss": False,
        "BF": BF,
    },
    {
        "name": "DFWSSNR", 
        "ylabel": r"FWSSNR Improvement $\Delta\text{FWSSNR}$ [dB]", 
        "title": "Source Extraction Performance",
        "latex_header": r"$\Delta$\gls{FWSSNR}$\uparrow$",
        "unit": r"\dB",
        "precision": 0,
        "is_min_better": False,
        "exclude_oracle_gss": False,
        "BF": BF,
    },    
    # {
    #     "name": "DSISDR", 
    #     "ylabel": r"SISDR Improvement $\Delta\text{SISDR}$ [dB]", 
    #     "title": "Source Extraction Performance",
    #     "latex_header": r"$\Delta$\gls{SISDR}$\uparrow$",
    #     "unit": r"\dB",
    #     "precision": 0,
    #     "is_min_better": False,
    #     "exclude_oracle_gss": False,
    #     "BF": BF,
    # },
    {
        "name": "DSDR", 
        "ylabel": r"SDR Improvement $\Delta\text{SDR}$ [dB]", 
        "title": "Source Extraction Performance",
        "latex_header": r"$\Delta$\gls{SDR}$\uparrow$",
        "unit": r"\dB",
        "precision": 0,
        "is_min_better": False,
        "exclude_oracle_gss": False,
        "BF": BF,
    },
    # {
    #     "name": "DHGSDR", 
    #     "ylabel": r"HGSDR Improvement $\Delta\text{HGSDR}$ [dB]", 
    #     "title": "Source Extraction Performance",
    #     "latex_header": r"$\Delta$\gls{HGSDR}$\uparrow$",
    #     "unit": r"\dB",
    #     "precision": 0,
    #     "is_min_better": False,
    #     "exclude_oracle_gss": False,
    #     "BF": BF,
    # },
    {
        "name": "STOIo", 
        "ylabel": r"$\text{STOI}$", 
        "title": "Source Extraction Performance",
        "latex_header": r"$\gls{STOI}$",
        "unit": r"",
        "precision": 2,
        "is_min_better": False,
        "exclude_oracle_gss": False,
        "BF": BF,
    },
    # {
    #     "name": "DPESQ", 
    #     "ylabel": r"PESQ Improvement $\Delta\text{PESQ}$", 
    #     "title": "Source Extraction Performance",
    #     "latex_header": r"$\Delta$\gls{PESQ}$\uparrow$",
    #     "unit": r"",
    #     "precision": 1,
    #     "is_min_better": False,
    #     "exclude_oracle_gss": False
    # },
]

# Style Settings
FIG_DIMENSIONS = setup_matplotlib_rc(n_metrics=len(METRICS), n_segments=len(SEGMENTS))
FORCE_Y_LIMITS = {
    # Optional manual limits: "DSINR": (0.4, 0.9)
}

# -----------------
# LaTeX Mappings & Captions
# -----------------
LATEX_LABELS = {
    "GSS": r"\gls{GSS}~\cite{horiguchi2021blockonlinegss}",
    "C": r"\gls{C}", "CSn": r"\gls{CSn}~\cite{markovich-golan_performance_2018}",
    "CWn": r"\gls{CWn}~\cite{markovich-golan_performance_2018}", "CSu": r"\gls{CSu}~\cite{markovich-golan_performance_2018}",
    "CWu": r"\gls{CWu}~\cite{markovich-golan_performance_2018}",
    "BOP": r"\gls{BOP}~\cite{cherkassky_successive_2020}", "BOP-S": r"\gls{BOP-S}*", "BOP-W": r"\gls{BOP-W}*",
    "BOPO": r"\gls{BOPO}*", "BOPO-S": r"\gls{BOPO-S}*", "BOPO-W": r"\gls{BOPO-W}*",
    "CB": r"\gls{CB}*", "CB-S": r"\gls{CB-S}*", "CB-W": r"\gls{CB-W}*", "CBW": r"\gls{CBW}~\cite{gode2023covariance}",
    "Oracle": r"Oracle"
}

CAPTIONS = {
    "OracleSAD": r"Median weighted Hermitian angle $\psi$ for the conventional and proposed \gls{RTF} vector estimation methods and median \gls{SINR} improvement for all considered methods for different \glspl{SNR} assuming oracle source activity knowledge.",
    "COSAD": r"Median weighted Hermitian angle $\psi$ and \gls{SINR} improvement for the conventional and proposed \gls{RTF} vector estimation methods for different \glspl{SNR} using the online source counting method~\cite{gode2026dnn}."
}


# ==============================================================================
# 3. EXECUTION
# ==============================================================================

# Output configuration
BASE_PLOT_FILENAME = "Klaus_PALD3D_cube"
PAPER_PLOT_DIR = Path("/data4/Henri/j3/framewiseSpeakerCounting/documents/Journal2/figures/Plots/revision_caused_by_J1_reviews")

for sadname in EXPERIMENTS:
    print(f"Generating Plots & Tables for {sadname}...")
    df_eval = dfs[sadname].reset_index(drop=True)

    # Setup paths
    fname = f"{BASE_PLOT_FILENAME}_{sadname}_2.pdf"
    path_local = resultsdirs[sadname] / "plots" / "J2_Generic" / fname
    path_paper = PAPER_PLOT_DIR / fname
    
    path_local.parent.mkdir(parents=True, exist_ok=True)
    path_paper.parent.mkdir(parents=True, exist_ok=True)
    output_files = [path_local, path_paper]

    # Generate Grid Figure
    plot_generic_grid(
        df=df_eval,
        metrics_config=METRICS,
        segments_config=SEGMENTS,
        group_by=GROUP_BY,
        xlabel_text=XLABEL,
        output_files=output_files,
        all_methods=ALL_METHODS,
        is_cosad=(sadname == "COSAD"),
        fig_dims=FIG_DIMENSIONS,
        force_y_limits=FORCE_Y_LIMITS
    )
    
    # Generate Table
    # generate_generic_latex_table(
    #     df=df_eval,
    #     sadname=sadname,
    #     metrics_config=METRICS,
    #     segments_config=SEGMENTS,
    #     all_methods=ALL_METHODS,
    #     method_labels=LATEX_LABELS,
    #     snr_levels=SNR_LEVELS,
    #     output_files=output_files,
    #     caption=CAPTIONS[sadname]
    # )
    print(f"-> Finished {sadname}. Saved to {path_local.parent}\n")

Generating Plots & Tables for OracleSAD...
-> Finished OracleSAD. Saved to /data4/Henri/j3/framewiseSpeakerCounting/results/J2_RUN/Klaus_PALD_3D_1/oracle/plots/J2_Generic

Generating Plots & Tables for COSAD...
-> Finished COSAD. Saved to /data4/Henri/j3/framewiseSpeakerCounting/results/J2_RUN/Klaus_PALD_3D_1/PrecomputedSAD/plots/J2_Generic

